# 02 — HNSW: O Algoritmo por Trás da Busca Vetorial Rápida

## O problema que o HNSW resolve

Busca de nearest neighbor em alta dimensão é computacionalmente custosa.

A abordagem ingênua — calcular a distância de uma query para todos os N vetores — é O(N). Com 1 milhão de vetores de 768 dimensões, isso significa 768 milhões de multiplicações por query. Em CPU, isso leva segundos.

Para um sistema RAG em produção recebendo centenas de queries por segundo, isso é inviável.

**HNSW (Hierarchical Navigable Small World)** resolve isso com uma estrutura de grafo hierárquica que permite busca *aproximada* em O(log N) — encontra os K vizinhos mais próximos com ~95-99% de acurácia, mas em milissegundos.

## Intuição do algoritmo

Imagine uma rede de metrô em camadas:
- **Camada superior (poucas estações)**: conexões longas de "salto rápido" — você chega perto do destino rapidamente
- **Camadas intermediárias**: refinamento progressivo
- **Camada inferior (todas as estações)**: busca local fina para encontrar os vizinhos exatos

Na busca, você começa no topo (saltos grandes, cobertura ampla) e desce progressivamente até a camada onde estão os vizinhos mais próximos do ponto de busca.

Esse design imita o conceito de "small world networks" — como as 6 conexões que separam qualquer pessoa de qualquer outra no mundo.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    HnswConfigDiff, OptimizersConfigDiff
)
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

client = QdrantClient(host='localhost', port=6333)
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Pronto!')

## 2.1 Os Parametros HNSW

### `m` — Numero de conexoes por no
- Controla a **densidade do grafo**
- Valores tipicos: 4 a 64
- Default Qdrant: 16
- **↑ m = melhor recall + mais RAM + indexacao mais lenta**

### `ef_construct` — Candidatos durante construcao
- Candidatos avaliados ao inserir cada no
- Valores tipicos: 50 a 500
- Default Qdrant: 100
- **↑ ef_construct = melhor qualidade do indice + indexacao mais lenta**

### `ef` — Candidatos durante busca (query time)
- Candidatos avaliados em cada busca
- Default: igual a `m * 2`
- **↑ ef = melhor recall + busca mais lenta**

In [ ]:
# Gerar dataset sintetico para experimentar
import random
random.seed(42)
np.random.seed(42)

n_docs = 2000
temas = ['machine learning', 'banco de dados', 'cloud computing', 'seguranca', 
         'frontend', 'backend', 'DevOps', 'data science']

docs_sinteticos = [
    f'{random.choice(temas)}: documento numero {i} sobre {random.choice(["fundamentos", "avancado", "pratico"])}'
    for i in range(n_docs)
]

print(f'Criando embeddings para {n_docs} documentos...')
t0 = time.time()
all_embs = model.encode(docs_sinteticos, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
print(f'Tempo: {time.time()-t0:.1f}s | Shape: {all_embs.shape}')

In [ ]:
## 2.1 Parâmetros do HNSW

O HNSW tem três parâmetros principais. Entender o que cada um faz é essencial para tunar o índice para o seu caso de uso:

### `m` — Número de conexões por nó
Cada ponto no grafo mantém `m` conexões com seus vizinhos. Mais conexões = grafo mais denso = busca mais precisa, mas mais memória e tempo de indexação.

- `m=4`: índice leve, busca rápida, recall menor (~90%)
- `m=16`: padrão do Qdrant, bom equilíbrio
- `m=32`: alta qualidade, 2x mais memória que m=16

### `ef_construct` — Qualidade da construção
Durante a indexação, o algoritmo considera `ef_construct` candidatos ao adicionar cada ponto. Maior valor = grafo de melhor qualidade = melhor recall, mas indexação mais lenta.

- `ef_construct=100`: padrão, bom para maioria dos casos
- `ef_construct=200+`: necessário apenas para datasets muito grandes ou alta precisão

### `ef` (search ef) — Candidatos na busca
Durante a query, considera `ef` candidatos. Aumentar melhora o recall sem exigir reindexação — você pode ajustar isso por query.

- `ef=128`: padrão, ~95% recall
- `ef=256+`: para quando você precisa de recall máximo e aceita latência maior

### Como os parâmetros afetam memória?

O grafo HNSW ocupa memória adicional além dos vetores em si. Cada conexão é armazenada como um ponteiro.

Memória aproximada do grafo: `N × m × 2 × 8 bytes` (links em ambas as direções)

Para 1M vetores com m=16: `1M × 16 × 2 × 8 = ~256MB` de overhead de grafo. Não é o custo dominante (os vetores em si custam muito mais), mas vale considerar para datasets extremamente grandes.

**Regra prática:** use os padrões (`m=16`, `ef_construct=100`) para datasets até 10M de vetores. Para datasets maiores ou qualidade crítica, considere aumentar `m` para 32 ou `ef_construct` para 200.

In [ ]:
# Visualizacao: tradeoff de configuracoes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ms = [c['m'] for c in configs_hnsw]
labels = [c['nome'] for c in configs_hnsw]
cores = ['#3498db', '#2ecc71', '#e74c3c']

# Memoria estimada (proporcional a m)
mem_estimada = [m * 4 for m in ms]  # bytes adicionais por ponto, aprox
axes[0].bar(labels, mem_estimada, color=cores)
axes[0].set_title('Memoria Estimada do Indice HNSW\n(proporcional a m)', fontweight='bold')
axes[0].set_ylabel('Memoria relativa')
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=20, ha='right', fontsize=9)

# Relacao recall estimado x velocidade (conceitual)
recall_estimado = [0.85, 0.95, 0.99]  # estimados
busca_ms = [float(r['Busca (ms/query)']) for r in resultados_hnsw]

for i, (rec, ms_val, label) in enumerate(zip(recall_estimado, busca_ms, labels)):
    axes[1].scatter(ms_val, rec, s=200, color=cores[i], label=label, zorder=3)
    axes[1].annotate(label, (ms_val, rec), xytext=(3, 3), textcoords='offset points', fontsize=8)

axes[1].set_title('Trade-off Recall vs Velocidade de Busca', fontweight='bold')
axes[1].set_xlabel('Tempo de Busca (ms/query)')
axes[1].set_ylabel('Recall Estimado')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impacto dos Parametros HNSW', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Guia de Configuracao

| Cenario | m | ef_construct | ef | RAM extra |
|---------|---|-------------|-----|----------|
| Prototipo / dev | 8 | 50 | 64 | Minima |
| Producao balanceado | 16 | 100 | 128 | Moderada |
| Alta qualidade | 32 | 200 | 256 | Alta |
| Recall maximo | 64 | 500 | 512 | Muito alta |

**Regra pratica:**
- Para colecoes < 100K docs: defaults sao perfeitos
- Para > 1M docs: aumente m e ef_construct gradualmente
- Sempre meca o recall antes de ir para producao!

## Proximo
- [03 — Quantizacao](03_quantization.html)

## Resumo: Quando tunar o HNSW?

Na maioria dos projetos, você **não precisa tunar o HNSW**. Os padrões do Qdrant funcionam bem para datasets de até alguns milhões de vetores.

Você deve considerar tuning quando:
- Dataset > 10M vetores: aumente `m` para melhor recall
- Latência crítica < 5ms: reduza `ef` na query (aceite recall menor)
- Precisão máxima necessária: aumente `ef_construct` e `ef`

| Situação | `m` | `ef_construct` | `ef` query |
|----------|-----|----------------|------------|
| Default / protótipo | 16 | 100 | 128 |
| Alta qualidade | 32 | 200 | 256 |
| Velocidade máxima | 8 | 100 | 64 |
| Dataset enorme | 16 | 100 | 128 + quantização |

**Próximos passos:**
- [03 — Quantization](03_quantization.html): como comprimir o índice para economizar memória sem perder recall